# Suite TXN — Transactions: mutation expressions over frames, and what they are built from

A transaction is a mutation expression over a stream of frames (docs/STREAMING_QUERIES.md; docs/OBJECT_IDENTITY.md,
Decided, 85 to 97): its terms are reference objects of the `Mutations` schemas, linked by relations, each term's
pattern a `Schemas.OfPattern` built through its link, and a term chosen through a kind union, `Mutations.OfTerm`. A
pattern's `spec` says what must hold of a frame, written in the builders' notation over the frame's symbols and recorded
as mutations. A keyed adjacency takes a mapping in place of the peer shorthand (FRAMEWORK.md, Resolved).

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import gc

from mbse.Schemas.Framework import Builders as B, Paths, Plain, Proposals, Schemas as S, Transactions, Visitors
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import instance_problems, raises


def crm(store):
    """crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by name; crm.Note, counted."""
    Int, Str = store.Schemas.OfInt, store.Schemas.OfStr
    with store.transaction():  # a fixture's schemas, one transaction
        Knows = store.Schemas.OfRelation(lambda r: r.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")])
                                         .properties({"since": Int}))
        Person = store.Schemas.OfObject(lambda o: o.name("crm.Person").ref(True).properties({"name": Str, "age": Int}))
        Holds = store.Schemas.OfRelation(lambda r: (
            r.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({"key": Str})
            .uniques([["book", "key"], ["contact"]])))
        store.Schemas.OfObject(Person, lambda o: o.adjacencies([
            lambda t: t.name("knows").relation([Knows]).me("knower"),
            lambda t: t.relation([Knows]).me("known"),
            lambda t: t.name("book").relation([Holds]).me("contact").owner(True),
        ]))
        Book = store.Schemas.OfObject(lambda o: o.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
            "contacts": lambda t: t.relation([Holds]).me("book")}))
        Note = store.Schemas.OfObject(lambda o: o.name("crm.Note").ref(True).properties({"text": Str}))
        Meets = store.Schemas.OfRelation(lambda r: r.name("crm.Meets").links([lambda t, n=n: t.name(n) for n in ("a", "b", "c")]))
        Kind = store.Schemas.OfUnion(lambda u: u.name("crm.Kind").branches([lambda t: t.name("person").schema([Person]),
                                                                             lambda t: t.name("note").schema([Note])]))
        Picks = store.Schemas.OfRelation(lambda r: r.name("crm.Picks").links([lambda t: t.name("owner"),
                                                                           lambda t: t.name("pick").schema([Kind])])
                                         .properties({"rank": Int}))
        Pins = store.Schemas.OfRelation(lambda r: r.name("crm.Pins").links([lambda t: t.name("note"),
                                                                         lambda t: t.name("book").schema([Book])])
                                        .properties({"at": Int}))
        Trio = store.Schemas.OfRelation(lambda r: r.links([lambda t, n=n: t.name(n) for n in ("a", "b", "c")]))  # unnamed
        Odd = store.Schemas.OfObject(lambda o: o.name("crm.Odd").ref(True).adjacencies([lambda t: t.name("x").me("a")]))  # no relation yet
        Odds = store.Schemas.OfRelation(lambda r: r.name("crm.Odds").links([lambda t: t.name("note"), lambda t: t.name("odd").schema([Odd])])
                                        .properties({"k": Int}))
        store.Schemas.OfObject(Person, lambda o: o.adjacencies({"meets": lambda t: t.relation([Meets]).me("a"),
                                                                "trio": lambda t: t.relation([Trio]).me("a")}))
        store.Schemas.OfObject(Note, lambda o: o.adjacencies({"picks": lambda t: t.relation([Picks]).me("owner"),
                                                              "pins": lambda t: t.relation([Pins]).me("note"),
                                                              "odds": lambda t: t.relation([Odds]).me("note")}))
    return Person, Knows, Book, Note


store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
Person, Knows, Book, Note = crm(store)

## TXN-01 · A kind union: a link's schema a union of reference object schemas, each branch a kind built through it

In [ ]:
Mutations = store.Mutations
OfTerm = Mutations.OfTerm.as_schema()
assert S._kinds(OfTerm) == {"map": Mutations.OfMap.as_schema(), "reduce": Mutations.OfReduce.as_schema(),
                            "partition": Mutations.OfPartition.as_schema(), "update": Mutations.OfUpdate.as_schema(),
                            "upsert": Mutations.OfUpsert.as_schema()}
assert S._kinds(Person) is None and S._kinds(store.Schemas.OfStr.as_schema()) is None
assert all(store.registered(name).validate() == [] for name in store.names() if name.startswith("Mutations."))
with store.transaction():
    mixed = store.Schemas.OfUnion(lambda u: u.branches([lambda t: t.name("p").schema([Person]),
                                                        lambda t: t.name("v").schema([store.Schemas.OfObject(lambda o: o)])]))
    assert store.Schemas.OfRelation(lambda r: r.links([lambda t: t.name("a"), lambda t: t.name("b").schema([mixed])])).validate() == [
        "link 'b': a kind union's branches are reference object schemas"]
# Building through the link: a branch, called, builds an object of its kind; the peer shorthand's visitor and an entry
# builder each have the branches as members.
tx = store.Mutations.OfTransaction(lambda t: t.uid("t-1").mutations([lambda mu: mu.upsert(lambda u: (
    u.key("k").then([lambda w: w.update(lambda d: d)])))]))
root = tx.mutations[0].term
assert root.as_schema() is Mutations.OfUpsert.as_schema() and str(root.key) == "k"
assert root.then[0].next.as_schema() is Mutations.OfUpdate.as_schema()
other = store.Mutations.OfTransaction(lambda t: t.uid("t-2").mutations(lambda a: a.insert(lambda e: e.map(lambda m: m))))
assert other.mutations[0].term.as_schema() is Mutations.OfMap.as_schema()
kept = store.Mutations.OfTransaction(lambda t: t.uid("t-3").mutations(lambda a: a.insert(lambda e: e.update(root.then[0].next))))
assert kept.mutations[0].term is root.then[0].next  # a kind given an object links it
with raises(TypeError, match="link 'term' is a kind union: build through one of its kinds, map, reduce, partition, update, upsert"):
    store.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.uid]))  # a member of no kind: which object?
with raises(TypeError, match="link 'term' is a kind union: build through one of its kinds"):
    store.Mutations.OfTransaction(lambda t: t.mutations(lambda a: a.insert(lambda e: e.term(lambda x: x))))
seen = []
store.Mutations.OfTransaction(lambda t: t.mutations(lambda a: a.insert(lambda e: seen.append(repr(e.upsert)) or e.upsert(lambda u: u))))
assert seen == ["<kind of link 'term'>"]

## TXN-02 · Mappings for keyed adjacencies: each item an entry, its key the one property, its value a callback or given to the one member that takes it

In [ ]:
assert list(Person.properties) == ["name", "age"] and Person.properties["age"].schema is Int  # the schema builders'
with store.transaction():
    Tagged = store.Schemas.OfObject(lambda o: o.properties({"tag": lambda t: t.schema([Str]).description("a label")}))
    assert Tagged.properties["tag"].description == "a label"
    with raises(TypeError, match="'schemas' takes a mapping only where its relation has two links and one property"):
        store.Schemas.OfAny(lambda a: a.schemas({"n": Int}))
    with raises(TypeError, match="more than one member of the entry takes 'text': give a callback"):
        store.Schemas.OfObject(lambda o: o.properties({"x": "text"}))  # its description and its naming pattern
    with raises(TypeError, match="no member of the entry takes 3: give a callback"):
        store.Schemas.OfObject(lambda o: o.properties({"x": 3}))
root_module = store.module()
with store.transaction():
    store.Schemas.OfModule(root_module, lambda m: m.schemas({"Tagged": Tagged}))  # the link alone takes a schema
assert root_module.schemas["Tagged"] is Tagged
book = store.singleton("crm.TheBook")
with store.transaction():
    store.crm.Book(book, lambda b: b.contacts({"ada": lambda e: e.name("Ada"), "bob": lambda e: e.name("Bob").age(36)}))
ada, bob = book.contacts({"key": "ada"}), book.contacts({"key": "bob"})  # the proxies': each a contact, by its key
assert [(str(e.key), str(e.contact.name)) for e in book.contacts] == [("ada", "Ada"), ("bob", "Bob")] and int(bob.age) == 36
with store.transaction():
    store.crm.Person(ada, lambda c: c.knows({2020: lambda e: e.known(bob)}))
assert [(int(e.since), e.known) for e in ada.knows] == [(2020, bob)]
with store.transaction():
    with raises(TypeError, match="more than one member of the entry takes <crm.Person>: give a callback"):
        store.crm.Person(ada, lambda c: c.knows({2021: bob}))  # the link, and the linked person's own 'knows'
    with raises(TypeError, match="no member of the entry takes 1.5: give a callback"):
        store.crm.Book(book, lambda b: b.contacts({"x": 1.5}))
with raises(TypeError, match="adjacency 'mutations' takes a mapping only where its relation has two links and one property"):
    store.Mutations.OfTransaction(lambda t: t.mutations({"x": lambda e: e}))
with store.transaction():
    with raises(TypeError, match="no member of the entry takes 'x': give a callback"):
        store.Schemas.OfModule(root_module, lambda m: m.schemas({"x": "x"}))
    with raises(TypeError, match="'schemas' takes an iterable of what it links, or of entries, a mapping, or a callback"):
        store.Schemas.OfAny(lambda a: a.schemas("x"))
# A link of a kind union, or of a schema it declares, takes an object of it; a callback chooses a kind.
with store.transaction():
    note = store.crm.Note(lambda n: n.picks({1: ada, 2: lambda e: e.note(lambda m: m.text("aside"))}).pins({3: book}))
assert [(int(e.rank), e.pick.as_schema().name) for e in note.picks] == [(1, "crm.Person"), (2, "crm.Note")]
assert [(int(e.at), e.book) for e in note.pins] == [(3, book)]
with store.transaction():
    with raises(TypeError, match="no member of the entry takes 'zz': give a callback"):
        store.crm.Note(lambda n: n.odds({1: "zz"}))  # an adjacency with no relation yet takes nothing
outside = B.OfObject.Builder().properties({"n": lambda t: t.description("d")}).create()  # outside a store too
assert outside.properties["n"].description == "d"

## TXN-03 · A transaction's expression: reference objects linked by relations, each term's pattern a schema object built through its link and held while its term is

In [ ]:
def upserting():
    return store.Mutations.OfTransaction(lambda t: t.uid("t-9").mutations([lambda mu: mu.upsert(lambda u: u.exists([
        lambda p: p.symbols({"ada": Person}).description("whom")]))]))


held = len(store._unnamed)
tx = upserting()
pattern = tx.mutations[0].term.exists[0].pattern
assert isinstance(pattern, S.OfPattern.Data) and list(pattern.symbols) == ["ada"] and pattern.description == "whom"
assert store.holds(pattern) and store.holds(pattern.symbols["ada"]) and len(store._unnamed) == held + 2
with raises(LookupError, match="a pattern with no name of its own"):
    Paths.of(pattern)
del tx, pattern
gc.collect()
assert len(store._unnamed) == held  # a term's pattern goes with its transaction (Decided, 94)
with store.transaction():
    Edit = store.Schemas.OfPattern(lambda p: p.name("crm.Edit").symbols({"ada": Person}))
store.Mutations.OfUpdate(lambda u: u.apply([lambda p: p.pattern(Edit).description("edits")]))  # linked, then updated
assert Edit.description == "edits"
assert store._meta_builder("crm.Person") is None and store._meta_builder("Schemas.Nope") is None
assert repr(store._meta_builder("Schemas.OfPattern")) == "<builder of a pattern>"
assert store._meta_builder_class(Edit) is B.PatternBuilder

## TXN-04 · A spec: written in the builders' notation over the frame's symbols, recorded as mutations, by symbol

In [ ]:
with store.transaction():  # the patterns its specs are written in
    def ref(name):
        return {"ref": {"symbol": name}}


    def path_of(target):
        return {"ref": {"path": Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(target))}}


    def link(name, target):
        return {"as_link": {"name": name, "mutations": [{"set": target}]}}


    def prop(name, value):
        return {"as_property": {"name": name, "mutations": [{"set": {"value": value}}]}}


    def entries(name, *items):
        return {"as_adjacency": {"name": name, "mutations": [{"insert": {"entries": [{"mutations": list(i)} for i in items]}}]}}


    Meet = store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person, "bob": Person}).spec([
        lambda _: _.bob.name("Bob").knows([_.ada]),
        lambda _: _.ada.name("Ada").age(None).knows([lambda e: e.known(_.bob).since(2020)]),
    ]))
    assert Meet.spec == (
        {"bob": [prop("name", "Bob"), entries("knows", [link("known", ref("ada"))])]},
        {"ada": [prop("name", "Ada"), {"as_property": {"name": "age", "mutations": [{"clear": {}}]}},
                 entries("knows", [link("known", ref("bob")), prop("since", 2020)])]},
    )
    book = store.singleton("crm.TheBook")
    Owned = store.Schemas.OfPattern(lambda p: p.symbols({"bob": Person}).spec(  # one callback; a path; a mapping
        lambda _: _.bob.book({"bob": book}).knows(lambda a: a.insert(lambda e: e.known(lambda n: n.name("Dan")).since(1)))))
    assert Owned.spec == ({"bob": [
        entries("book", [link("book", path_of(book)), prop("key", "bob")]),
        entries("knows", [link("known", {"def": {"schema": "crm.Person", "mutations": [prop("name", "Dan")]}}), prop("since", 1)]),
    ]},)
    Defines = store.Schemas.OfPattern(lambda p: p.symbols({"P": store.Schemas.OfObject}).spec([
        lambda _: _.P.name("crm.Pet").ref(True).properties({"name": Str, "legs": lambda t: t.schema([Int])}),
    ]))
    property_schema = lambda schema: {"def": {"schema": "Schemas.OfProperty", "mutations": [
        entries("schema", [link("schema", path_of(schema))])]}}
    assert Defines.spec == ({"P": [prop("name", "crm.Pet"), prop("ref", True), entries(
        "properties", [link("property", property_schema(Str)), prop("name", "name")],
        [link("property", property_schema(Int)), prop("name", "legs")])]},)
    Terms = store.Schemas.OfPattern(lambda p: p.symbols({"t": store.Mutations.OfTransaction}).spec([
        lambda _: _.t.uid("u").mutations([lambda e: e.upsert(lambda u: u.key("k"))]),  # a kind union: a kind defined
        lambda _: _.t.mutations([lambda e: e.term(_.t)]),
    ]))
    assert Terms.spec[0] == {"t": [prop("uid", "u"), entries("mutations", [link("term", {"def": {
        "schema": "Mutations.OfUpsert", "mutations": [prop("key", "k")]}})])]}
    assert Terms.spec[1] == {"t": [entries("mutations", [link("term", ref("t"))])]}
    # Reading through the frame, and what a spec refuses.
    seen = []
    store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person}).spec([
        lambda _: seen.extend([repr(_), repr(_.ada), _.ada.name.get(), _.ada.as_schema() is Person,
                               _.nope.as_schema(), _.ada.nope.as_schema(), _.ada.knows.as_schema().name,
                               hasattr(_, "_hidden"), hasattr(_.ada, "_hidden"), hasattr(_.ada.name, "_hidden")]),
        lambda _: _.ada.knows(lambda a: a.insert(lambda e: seen.extend([
            e.as_schema() is Knows.entry, hasattr(e, "_hidden"), e.nope.as_schema()]) or e.known(_.ada))),
        lambda _: _.ada.knows([lambda e: e.known(lambda n: seen.append(repr(n)))]),
    ]))
    assert seen == ["<frame>", "<spec of ada>", None, True, None, None, "knows", False, False, False, True, False, None,
                    "<spec of an object>"]
    Given = store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person, "bob": Person}).spec([
        lambda _: _.bob.knows({2020: "Dan"}).book({"b": _.ada}),  # a property of the linked object; an adjacency of it
        lambda _: _.bob.knows([lambda e: e.known(lambda n: n.name("Eve")).age(3)]),  # the object it defines, written on
    ]))
    assert Given.spec[0] == {"bob": [
        entries("knows", [link("known", {"def": {"schema": "crm.Person", "mutations": [prop("name", "Dan")]}}), prop("since", 2020)]),
        entries("book", [link("book", {"def": {"schema": "crm.Book", "mutations": [entries("contacts", [link("contact", ref("ada"))])]}}),
                         prop("key", "b")])]}
    assert Given.spec[1] == {"bob": [entries("knows", [link("known", {"def": {"schema": "crm.Person", "mutations": [
        prop("name", "Eve"), prop("age", 3)]}})])]}
    note = store.crm.Note(lambda n: n)
    for spec, error, says in (
        (lambda _: _.x.name("x"), TypeError, "symbol 'x' has no schema: a spec writes what its object's schema declares"),
        (lambda _: _.ada.knows(3), TypeError, "adjacency 'knows' takes an iterable of entries or of what it links, a mapping"),
        (lambda _: _.ada.book({"a": 1.5}), TypeError, "no member of the entry takes 1.5: give a callback"),
        (lambda _: _.ada.knows([lambda e: e.known(_.ada).name("Z")]), ValueError,
         "link 'known' links an object a spec does not define: write its members in its own spec"),
        (lambda _: _.ada.knows([lambda e: e.known(note)]), ValueError,
         "a counted 'crm.Note' has no path: a spec refers to it through a symbol"),
        (lambda _: _.ada.meets([lambda e: e]), TypeError,
         "adjacency 'meets': entries of crm.Meets have more than two links; give each entry's links"),
        (lambda _: _.ada.trio([lambda e: e]), TypeError, "adjacency 'trio': entries of its relation have more than two links"),
        (lambda _: _.ada.knows({2020: _.ada}), TypeError, "more than one member of the entry takes <spec of ada>: give a callback"),
        (lambda _: _.ada.meets({1: _.ada}), TypeError,
         "adjacency 'meets' takes a mapping only where its relation has two links and one property"),
        (lambda _: _.ada.knows([lambda e: e.known(lambda n: n.knows([lambda f: f.known(n)]))]), ValueError,
         "an object a spec defines is linked where it is defined"),
        (lambda _: _.t.mutations([lambda e: e.term(lambda x: x)]), TypeError,
         "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"),
        (lambda _: _.t.mutations([lambda e: e.uid("u")]), TypeError,
         "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"),
    ):
        with raises(error, match=says):
            store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person, "x": lambda t: t,
                                                         "t": store.Mutations.OfTransaction}).spec([spec]))
    entry = []
    store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person}).spec([lambda _: _.ada.knows([lambda e: entry.append(e) or e.known(_.ada)])]))
    with raises(AttributeError):
        entry[0].since = 1  # a member is written by calling it
    with raises(TypeError, match="a spec is written in a store, whose schemas its symbols name"):
        B.PatternBuilder().spec([lambda _: _])
    with raises(ReadOnlyError):
        B.PatternBuilder(Meet, False, store).spec([lambda _: _])

## TXN-05 · A transaction's proposals: data written apart from the store until it commits, dropped where it raises; a schema object that existed changed at its commit

In [ ]:
store5 = B.Store()
Int5, Str5 = store5.Schemas.OfInt, store5.Schemas.OfStr
Person5, _, Book5, Note5 = crm(store5)
book5 = store5.singleton("crm.TheBook")
with store5._transacting():
    store5.crm.Book(book5, lambda b: b.contacts({"ada": lambda e: e.name("Ada")}))
    ada5 = book5.contacts({"key": "ada"})
    note5 = store5.crm.Note(lambda n: n.text("x"))
    store5.crm.Note(note5, lambda n: n.text("y"))
    assert str(ada5.name) == "Ada" and str(note5.text) == "y"  # read through its proposals
assert [str(e.key) for e in book5.contacts] == ["ada"] and str(note5.text) == "y"  # committed
with raises(ValueError, match="stop"):
    with store5._transacting():
        store5.crm.Book(book5, lambda b: b.contacts({"bob": lambda e: e.name("Bob")}))
        store5.crm.Person(ada5, lambda c: c.name("Ada L."))
        store5.Schemas.OfObject(lambda o: o.name("crm.Gone").ref(True))
        temp5 = store5.Schemas.OfObject(lambda o: o.ref(True))  # no name of its own, then one
        store5.Schemas.OfObject(temp5, lambda o: o.name("crm.Temp"))
        raise ValueError("stop")
assert [str(e.key) for e in book5.contacts] == ["ada"] and str(ada5.name) == "Ada"  # dropped
assert "crm.Gone" not in store5.names() and "Gone" not in store5.module("crm").schemas and not store5.holds(temp5)
with store5._transacting():  # a schema object that existed: changed when the transaction commits
    store5.Schemas.OfObject(Note5, lambda o: o.properties({"tag": Str5}))
    assert "tag" not in Note5.properties
    store5.Schemas.OfObject(Note5, lambda o: o.properties({"mood": Str5}))  # from what the one before made
    Fresh = store5.Schemas.OfObject(lambda o: o.name("crm.Fresh").ref(True))
    store5.Schemas.OfObject(Fresh, lambda o: o.properties({"n": Int5}))  # one it made: changed at once
    assert list(Fresh.properties) == ["n"]
assert list(Note5.properties) == ["text", "tag", "mood"]
with raises(ValueError, match="schema 'crm.Book' is already registered"):
    with store5._transacting():  # refused when it commits: what it wrote is undone, the schemas' and the data's
        store5.Schemas.OfObject(Person5, lambda o: o.description("people"))
        store5.crm.Note(note5, lambda n: n.text("z"))
        store5.Schemas.OfObject(Note5, lambda o: o.name("crm.Book"))
assert Person5.description is None and Note5.name == "crm.Note" and str(note5.text) == "y"
assert store5.registered("crm.Note") is Note5 and store5.registered("crm.Book") is Book5
with raises(ValueError, match="a transaction is open: transactions do not nest"):
    with store5._transacting():
        with store5._transacting():
            pass

## TXN-06 · store.transaction: a fixture upserted in one transaction, its frames given back; run again, it finds what it made, an owned object by its spec, a counted one by its frame, key and symbol

In [ ]:
store6 = B.Store()
Person6, Knows6, Book6, Note6 = crm(store6)
book6 = store6.singleton("crm.TheBook")


def fixture(t):
    return t.uid("fixture").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
        {"ada": Person6, "bob": Person6, "book": Book6, "note": Note6}).spec([
            lambda _: _.book.contacts({"ada": lambda e: e.contact(_.ada), "bob": lambda e: e.contact(_.bob)}),
            lambda _: _.ada.name("Ada").knows([lambda e: e.known(_.bob).since(2020)]),
            lambda _: _.bob.name("Bob").age(None),
            lambda _: _.note.text("hi").pins({1: _.book}).odds([lambda e: e.odd(lambda o: o).k(7)]),  # an Odd defined
        ])]))])


frames = store6.transaction(fixture)
assert len(frames) == 1 and list(frames[0]) == ["ada", "bob", "book", "note"] and frames[0]["book"] is book6
ada6, bob6, note6 = frames[0]["ada"], frames[0]["bob"], frames[0]["note"]
assert [(str(e.key), str(e.contact.name)) for e in book6.contacts] == [("ada", "Ada"), ("bob", "Bob")]
assert [(e.known, e.since.get()) for e in ada6.knows] == [(bob6, 2020)]
assert [(e.book, e.at.get()) for e in note6.pins] == [(book6, 1)] and str(note6.text) == "hi"
odd6 = note6.odds[0].odd
assert odd6.as_schema() is store6.registered("crm.Odd") and note6.odds[0].k.get() == 7
again = store6.transaction(fixture)  # found, not made again: nothing to write
assert [again[0][name] for name in ("ada", "bob", "note")] == [ada6, bob6, note6] and len(book6.contacts) == 2
assert len(ada6.knows) == 1 and len(note6.pins) == 1 and [e.odd for e in note6.odds] == [odd6]
assert store6.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": Person6}).spec([lambda _: _.x.book({"ada": book6})])]))])) == [{"x": ada6}]  # the book by its path
built = store6.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.key("other").exists([
    lambda p: p.symbols({"note": Note6}).spec([lambda _: _.note.text("hi")])]))]))  # built first, then executed
other = store6.transaction(built)[0]["note"]
assert other is not note6 and store6.transaction(built)[0]["note"] is other  # a key of its own: another note, kept
assert store6.transaction(lambda t: t) == [{}]  # no term: the one empty frame
assert store6.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([
    lambda p: p.symbols({"book": Book6})]))])) == [{"book": book6}]  # a singleton's symbol binds it

## TXN-07 · Map and update: a map binds each object its spec matches, one frame each, the frames before it multiplied; an update writes its spec over each frame

In [ ]:
store7 = B.Store()
Person7, Knows7, Book7, Note7 = crm(store7)
book7 = store7.singleton("crm.TheBook")
store7.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"a": Person7, "b": Person7, "c": Person7, "book": Book7}).spec([
        lambda _: _.book.contacts({"a": lambda e: e.contact(_.a), "b": lambda e: e.contact(_.b),
                                   "c": lambda e: e.contact(_.c)}),
        lambda _: _.a.name("Ada").age(36).knows([lambda e: e.known(_.b).since(2020), lambda e: e.known(_.c).since(2021)]),
        lambda _: _.b.name("Bob").age(36).knows([_.c]),
        lambda _: _.c.name("Cy"),
    ])]))]))


def mapped(*specs, symbols=None, then=None):
    term = lambda m: m.symbols([lambda p: p.symbols(symbols or {"x": Person7, "y": Person7}).spec(list(specs))])
    return store7.transaction(lambda t: t.mutations([lambda mu: mu.map(
        term if then is None else lambda m: term(m).then([then]))]))


def names(frames):
    return [tuple(str(target.name) for target in frame.values()) for frame in frames]


assert names(mapped(lambda _: _.x.knows([_.y]))) == [("Ada", "Bob"), ("Ada", "Cy"), ("Bob", "Cy")]
assert names(mapped(lambda _: _.x.knows([lambda e: e.known(lambda k: k.name("Bob").knows([_.y]))]))) == [("Ada", "Cy")]  # through one
assert names(mapped(lambda _: _.x.knows([lambda e: e.known(_.y).since(2021)]))) == [("Ada", "Cy")]
assert names(mapped(lambda _: _.x.age(36), symbols={"x": Person7})) == [("Ada",), ("Bob",)]
about = mapped(lambda _: _.x.name("Ada"), symbols={"x": Person7}, then=lambda w: w.upsert(lambda u: u.exists([
    lambda p: p.symbols({"x": Person7, "n": Note7}).spec([lambda _: _.n.text("about Ada")])])))
assert [(str(f["x"].name), str(f["n"].text)) for f in about] == [("Ada", "about Ada")]  # x as the map bound it
assert names(mapped(lambda _: _.x.age(None), symbols={"x": Person7})) == [("Cy",)]  # a property cleared: none held
assert names(mapped(lambda _: _.x.name("Nobody"), symbols={"x": Person7})) == []
assert len(mapped(symbols={"x": Person7, "y": Person7})) == 9  # no spec: the cross product
assert [(str(f["x"].name), f["y"]) for f in mapped(lambda _: _.x.book({"b": _.y}), symbols={"x": Person7, "y": Book7})] \
    == [("Bob", book7)]  # an entry's property, through a mapping
store7.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": Person7}).spec([lambda _: _.x.age(36)])]).then([lambda w: w.update(lambda u: u.apply([lambda p: p.symbols(
        {"x": Person7}).spec([lambda _: _.x.age(37)])]))]))]))
assert sorted((str(p.name), store7.member(p, "age")) for p in store7.extent("crm.Person")) == [
    ("Ada", 37), ("Bob", 37), ("Cy", None)]
store7.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": Person7, "bk": Book7}).spec([lambda _: _.x.name("Bob")])]).then([lambda w: w.update(lambda u: u.apply([
        lambda p: p.symbols({"x": Person7, "bk": Book7}).spec([lambda _: _.x.age(None).book({"robert": _.bk})])]))]))]))
bob7 = book7.contacts({"key": "robert"})  # its one owning entry replaced: its key now "robert"
assert str(bob7.name) == "Bob" and store7.member(bob7, "age") is None and [str(e.key) for e in book7.contacts] == [
    "a", "c", "robert"]

## TXN-08 · Reduce and partition: a reduce drops symbols, keeping each frame left once; a partition runs its first, last or every branch, or none where it has no branch, and joins what they end with

In [ ]:
pairs = lambda m: m.symbols([lambda p: p.symbols({"x": Person7, "y": Person7}).spec([lambda _: _.x.knows([_.y])])])
dropping = lambda names_: lambda r: r.symbols([lambda c: c.pattern([lambda p: p.symbols({"x": Person7, "y": Person7})])
                                              .symbols(names_)])


def run(then):
    return store7.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: pairs(m).then([then]))]))


assert names(run(lambda w: w.reduce(dropping(["y"])))) == [("Ada",), ("Bob",)]  # three frames, two left
assert names(run(lambda w: w.reduce(dropping(["x"])))) == [("Cy",), ("Bob",)]  # in the order the store reaches them
labelled = lambda text: lambda e: e.branch(lambda b: b.then([lambda w: w.upsert(lambda u: u.key(text).exists([
    lambda p: p.symbols({"n": Note7}).spec([lambda _: _.n.text(text)])]))]))
arms = lambda p: p.branches([labelled("one"), labelled("two")])
partitioned = lambda first=False, last=False, branches=arms, then=None: run(lambda w: w.partition(
    lambda p: (lambda q: q if then is None else q.then([then]))(branches(p.first(first).last(last)))))
texts = lambda frames: [str(frame["n"].text) for frame in frames]
assert texts(partitioned()) == ["one"] * 3 + ["two"] * 3  # every branch, each frame through each
assert texts(partitioned(first=True)) == ["one"] * 3 and texts(partitioned(last=True)) == ["two"] * 3
otherwise = lambda p: p.none([lambda w: w.reduce(dropping(["y"]))])
assert names(partitioned(branches=otherwise)) == [("Ada",), ("Bob",)]  # no branch: its none
assert partitioned(branches=lambda p: p) == []  # neither: its frames end
joined = partitioned(then=lambda w: w.reduce(dropping(["y", "n"])))  # what the branches end with, joined, then on
assert names(joined) == [("Ada",), ("Bob",)]
assert len(run(lambda w: w.partition(lambda p: p.branches([lambda e: e.branch(lambda b: b)])))) == 3  # a branch with no then: as it is

## TXN-09 · What a transaction refuses before it runs (a symbol no term before binds, an expression, a pattern's variables, a symbol of no reference object schema), and a transaction that raises leaving the store as it was

In [ ]:
store9 = B.Store()
Person9, Knows9, Book9, Note9 = crm(store9)
Int9 = store9.Schemas.OfInt
with store9.transaction():
    Exprs = store9.Schemas.OfExpression(lambda e: e.name("crm.True").pattern([lambda p: p]).result([Int9]))
person = lambda p: p.symbols({"x": Person9})


def refused(term, says):
    with raises(ValueError, match=says):
        store9.transaction(lambda t: t.mutations([term]))


refused(lambda mu: mu.update(lambda u: u.apply([person])), "symbol 'x' of an update is bound by no term before it")
refused(lambda mu: mu.reduce(lambda r: r.symbols([lambda c: c.pattern([person]).symbols(["x"])])),
        "symbol 'x' of a reduce is bound by no term before it")
refused(lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": Person9}).requires([Exprs])])),
        "a map's pattern requires constraints: transactions evaluate no expressions until step 4h")
refused(lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.variables({"v": Int9})])),
        "an upsert's pattern has variables")
refused(lambda mu: mu.map(lambda m: m.variables([person])), "a map of variables")
refused(lambda mu: mu.map(lambda m: m.symbols([person]).then([lambda w: w.reduce(lambda r: r.symbols([
    lambda c: c.pattern([person]).symbols(["x"]).suffixes({"current": "_a", "next": "_b"})]))])),
    "a reduce that collects")
refused(lambda mu: mu.partition(lambda p: p.first(True).last(True)),
        "a partition dispatches to its first matching branch or its last, not both")
refused(lambda mu: mu.partition(lambda p: p.branches([lambda e: e.branch(lambda b: b.requires([Exprs]))])), "a branch requires constraints")
refused(lambda mu: mu.map(lambda m: m), "a map takes one pattern in 'symbols', not 0")
refused(lambda mu: mu.reduce(lambda r: r), "a reduce takes one collector in 'symbols', not 0")
refused(lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols({"s": Int9})])),
        "symbol 's' binds no reference object of one schema")
refused(lambda mu: mu.map(lambda m: m.symbols([person]).then([lambda w: w.partition(lambda p: p.branches([
    lambda e: e.branch(lambda b: b.then([lambda w2: w2.upsert(lambda u: u.exists([lambda q: q.symbols({"y": Person9})]))])),
    lambda e: e.branch(lambda b: b),
]).then([lambda w2: w2.update(lambda u: u.apply([lambda q: q.symbols({"y": Person9})]))]))])),
    "symbol 'y' of an update is bound by no term before it")  # bound in one branch only
book9 = store9.singleton("crm.TheBook")
ada9 = store9.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"ada": Person9, "book": Book9}).spec([lambda _: _.book.contacts({"ada": lambda e: e.contact(_.ada)}),
                                           lambda _: _.ada.name("Ada")])]))]))[0]["ada"]
orphaning = lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": Person9}).spec([
    lambda _: _.x.name("Ada")])]).then([lambda w: w.update(lambda u: u.apply([lambda p: p.symbols({"x": Person9}).spec([
        lambda _: _.x.age(1).knows([lambda e: e.known(_.x)])])]).then([lambda w2: w2.upsert(lambda u: u.exists([
            lambda p: p.symbols({"o": Person9}).spec([lambda _: _.o.name("Orphan")])]))]))]))])
with raises(ValueError, match="a 'crm.Person' needs an owner"):  # refused when the transaction ends
    store9.transaction(orphaning)
assert store9.member(ada9, "age") is None and len(ada9.knows) == 0 and [p for p in store9.extent("crm.Person")] == [ada9]
store9.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"bea": Person9, "book": Book9}).spec([lambda _: _.book.contacts({"bea": lambda e: e.contact(_.bea.name("Bea"))})])]))]))
knowing = store9.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([
    lambda p: p.symbols({"x": Person9}).spec([lambda _: _.x.name("Bea")])]).then([lambda w: w.update(lambda u: u.apply([
        lambda p: p.symbols({"x": Person9}).spec([lambda _: _.x.knows([ada9])])]))]))]))  # Ada by her path
with store9.transaction():
    store9.crm.Book(book9, lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("ada"))))  # Ada deleted
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    store9.transaction(knowing)
assert [str(e.contact.name) for e in book9.contacts] == ["Bea"]

## TXN-10 · Schema objects in transactions: a module's install upserts its schemas, found by name, created where none is, a schema that existed extended when the transaction commits

In [ ]:
store10 = B.Store()
Int10, Str10 = store10.Schemas.OfInt, store10.Schemas.OfStr
OfObject, OfRelation = store10.Schemas.OfObject, store10.Schemas.OfRelation


def install(t):
    """zoo's install (SCHEMA_EXTENSION.md, Injection): its schemas, upserted by name."""
    return t.uid("zoo-1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
        {"Pet": OfObject, "Likes": OfRelation}).spec([
            lambda _: _.Pet.name("zoo.Pet").ref(True).properties({"name": Str10, "legs": Int10}),
            lambda _: _.Likes.name("zoo.Likes").links([lambda e: e.link(lambda k: k.schema([_.Pet])).name("fan"),
                                                       lambda e: e.link(lambda k: k.schema([_.Pet])).name("idol")]),
            lambda _: _.Pet.adjacencies({"likes": lambda a: a.adjacency(lambda j: j.relation([_.Likes]).me("fan"))}),
        ])]))])


installed = store10.transaction(install)[0]
Pet = store10.registered("zoo.Pet")
assert installed == {"Pet": Pet, "Likes": store10.registered("zoo.Likes")} and Pet.ref and list(Pet.properties) == [
    "name", "legs"] and Pet.properties["legs"].schema is Int10.as_schema()
assert list(store10.registered("zoo.Likes").entry.links) == ["fan", "idol"] and Pet.adjacencies["likes"].me == "fan"
assert store10.transaction(install)[0] == installed and list(Pet.properties) == ["name", "legs"]  # found: as it was
assert store10.module("zoo").schemas["Pet"] is Pet
store10.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"Pet": OfObject}).spec([lambda _: _.Pet.name("zoo.Pet").description("a pet").properties({"age": Int10})])]))]))
assert Pet.description == "a pet" and list(Pet.properties) == ["name", "legs", "age"]  # extended, by its name
rex = store10.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"rex": Pet, "tom": Pet}).spec([lambda _: _.rex.name("Rex").legs(4).likes([_.tom]), lambda _: _.tom.name("Tom")])]))]))
assert str(rex[0]["rex"].name) == "Rex" and [str(e.idol.name) for e in rex[0]["rex"].likes] == ["Tom"]  # its data
named = store10.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"S": OfObject}).spec([lambda _: _.S.ref(True)])]))]))
assert [frame["S"] for frame in named] == [Pet]  # the store's schemas, the framework's left out
assert [frame["S"].name for frame in store10.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([
    lambda p: p.symbols({"S": OfObject}).spec([lambda _: _.S.properties({"legs": Int10})])]))]))] == ["zoo.Pet"]
with raises(ValueError, match="schema 'zoo.Pet' is already registered"):  # a relation can not take a schema's name
    store10.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
        {"New": OfObject, "Clash": OfRelation}).spec([lambda _: _.New.name("zoo.New").ref(True),
                                                      lambda _: _.Clash.name("zoo.Pet")])]))]))
assert "zoo.New" not in store10.names() and "New" not in store10.module("zoo").schemas

## TXN-11 · A transaction's imperative scope: builders write within it, all or nothing; outside one a write is refused, but an expression's, the store's making and a decoder's, which joins the transaction open

In [ ]:
store11 = B.Store()  # its root module written as it is made
with raises(ValueError, match="a store is written only within a transaction (store.transaction)"):
    store11.Schemas.OfObject(lambda o: o.name("crm.Pet").ref(True))
with store11.transaction():
    Pet11 = store11.Schemas.OfObject(lambda o: o.name("crm.Pet").ref(True).properties({"n": store11.Schemas.OfInt}))
    rex11 = store11.crm.Pet(lambda p: p.n(1))
    assert store11.member(rex11, "n") == 1  # read within it
with raises(ValueError, match="written only within a transaction"):
    store11.crm.Pet(rex11, lambda p: p.n(2))
with raises(ValueError, match="stop"):
    with store11.transaction():
        store11.crm.Pet(rex11, lambda p: p.n(3))
        raise ValueError("stop")
assert store11.member(rex11, "n") == 1  # all or nothing
tx11 = store11.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([
    lambda p: p.symbols({"p": Pet11}).spec([lambda _: _.p.n(5)])]))]))  # an expression, built outside one (94)
assert store11.member(store11.transaction(tx11)[0]["p"], "n") == 5
plain11 = Plain.ToPlain(store11)(Pet11, rex11)
assert store11.member(Plain.FromPlain(store11)(Pet11, plain11), "n") == 1  # a decoder: a transaction of its own
other11 = B.Store()
with store11.transaction():
    assert store11.member(Plain.FromPlain(store11)(Pet11, plain11), "n") == 1  # or the one open, joined
    with raises(ValueError, match="written only within a transaction"):  # another store's: not this one's
        other11.Schemas.OfObject(lambda o: o.name("crm.Pet").ref(True))
    with other11.transaction():  # another store's transaction, open at once: its own (Decided, 99)
        Other11 = other11.Schemas.OfObject(lambda o: o.name("crm.Other").ref(True).properties({"n": other11.Schemas.OfInt}))
        other11.crm.Other(lambda o: o.n(1))
    assert other11.registered("crm.Other") is Other11  # committed when its own transaction ended
with raises(ValueError, match="stop"):
    with store11.transaction():
        store11.crm.Pet(rex11, lambda p: p.n(7))
        with other11.transaction():
            kept11 = other11.crm.Other(lambda o: o.n(2))
        raise ValueError("stop")
assert store11.member(rex11, "n") == 1 and other11.member(kept11, "n") == 2  # each all or nothing on its own
with raises(ValueError, match="stop"):
    with other11.transaction():
        other11.crm.Other(kept11, lambda o: o.n(3))
        with store11.transaction():
            store11.crm.Pet(rex11, lambda p: p.n(8))
        raise ValueError("stop")
assert store11.member(rex11, "n") == 8 and other11.member(kept11, "n") == 2
with raises(ValueError, match="a transaction is open: transactions do not nest"):
    with store11.transaction():
        with store11.transaction():
            pass

## TXN-12 · Proposals: the store as a transaction has it, its writes held apart until it commits; within a transaction, a savepoint, a decoder's, which fails alone

In [ ]:
store12 = B.Store()
with store12.transaction():
    Pet12 = store12.Schemas.OfObject(lambda o: o.name("crm.Pet").ref(True).singleton("crm.Rex").properties({"n": store12.Schemas.OfInt}))
    rex12 = store12.singleton("crm.Rex")
    store12.crm.Pet(rex12, lambda p: p.n(1))
plain12 = Plain.ToPlain(store12)(Pet12, rex12)
with store12.transaction():
    proposals12 = store12._proposals  # the store as the transaction has it: its writes, apart until it commits
    assert isinstance(proposals12, Proposals.OfStore) and instance_problems(proposals12, Visitors.OfStore) == []
    assert proposals12.schema("crm.Pet") is proposals12.registered("crm.Pet") is Pet12 and proposals12.name_of(Pet12) == "crm.Pet"
    assert list(proposals12.names()) == list(store12.names()) and not proposals12.framework(Pet12)
    assert proposals12.singleton("crm.Rex") is rex12 and list(proposals12.extent("crm.Pet")) == [rex12]
    assert proposals12.member(rex12, "n") == 1 and proposals12.builder("crm.Pet") is not None
    store12.crm.Pet(rex12, lambda p: p.n(2))
    assert proposals12.values(rex12) is not store12._committed.values(rex12)  # written: the proposals' copy
    with raises(ValueError, match="stop"):
        with store12._joined():  # within a transaction, a savepoint: proposals over its proposals, dropped alone
            assert store12._proposals.base is proposals12
            store12.crm.Pet(rex12, lambda p: p.n(3))
            raise ValueError("stop")
    assert store12.member(rex12, "n") == 2
    with raises(DecodeError, match="expected int, got str"):  # a decoder's: the transaction unaffected where it fails
        Plain.FromPlain(store12)(Pet12, {**plain12, "objects": [{**plain12["objects"][0], "properties": {"n": "x"}}]})
    assert store12.member(rex12, "n") == 2
    with store12._joined():  # committed into the transaction's proposals
        store12.Schemas.OfObject(Pet12, lambda o: o.description("a pet"))  # a schema that existed: waits
        store12.Schemas.OfObject(Pet12, lambda o: o.description("the pet"))  # from what the one before made
        store12.crm.Pet(rex12, lambda p: p.n(4))
    assert Pet12.description is None and store12.member(rex12, "n") == 4
    assert store12._committed.values(rex12) is not proposals12.values(rex12)
assert Pet12.description == "the pet" and store12.member(rex12, "n") == 4  # the transaction's commit